In [2]:
import subprocess
result = subprocess.run(['hdfs', 'dfs', '-ls', '/'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

FileNotFoundError: [Errno 2] No such file or directory: 'hdfs'

In [3]:
result = subprocess.run(['which', 'hdfs'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HDFSTest") \
    .config("spark.driver.memory", "2g") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020") \
    .getOrCreate()

df = spark.read.parquet('file:///home/jovyan/data/features/user_features')
df.write.parquet('hdfs://namenode:8020/instacart/features/user_features', mode='overwrite')
print('user_features migrated!')
for name in ['product_features', 'up_features', 'final_dataset']:
    df = spark.read.parquet(f'file:///home/jovyan/data/features/{name}')
    df.write.parquet(f'hdfs://namenode:8020/instacart/features/{name}', mode='overwrite')
    print(f'{name} migrated!')

user_features migrated!
product_features migrated!
up_features migrated!
final_dataset migrated!


In [5]:
df = spark.read.parquet('hdfs://namenode:8020/instacart/features/user_features')
df.write.parquet('hdfs://namenode:8020/instacart/features/user_features_v2', mode='overwrite')

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CheckResults") \
    .config("spark.driver.memory", "8g") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
    .config("spark.sql.catalog.hadoop_catalog.warehouse", "hdfs://namenode:8020/iceberg_warehouse") \
    .getOrCreate()

print(spark.table("hadoop_catalog.instacart.user_features").count())      # expect 206209
print(spark.table("hadoop_catalog.instacart.product_features").count())   # expect 49688 or close
print(spark.table("hadoop_catalog.instacart.up_features").count())
print(spark.table("hadoop_catalog.instacart.final_dataset").groupBy('label').count().show())
spark.sql("SHOW TABLES IN hadoop_catalog.instacart").show(100, False)

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `hadoop_catalog`.`instacart`.`user_features` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.;
'UnresolvedRelation [hadoop_catalog, instacart, user_features], [], false
